# 🎯 [AMM02_M03] 교전 일수 기반 1차 탄약/유류 기본 소모량 계산 Baseline 알고리즘 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/AMM02/amm02_m03_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 탄약·특수물자 (Level 3) |
| **미션 ID** | AMM02_M03 |
| **미션 주제** | 교전 일수 기반 1차 탄약/유류 기본 소모량 계산 Baseline 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
탄약 저장고는 공간이 한정되어 있고 품목마다 함께 둘 수 없는 제약이 있어, 앞으로 얼마나 소모되고 얼마나 자주 반출될지를 알아야 입고 위치와 재보급 순서를 합리적으로 정할 수 있다. 그러나 작전 단계별 고정 기준 소모율(CSR)만으로 계획하면 교전 강도가 낮은 날은 과잉 재고가 쌓이고 높은 날은 긴급 재보급이 반복되어 저장 공간과 작업 동선이 낭비된다. 본 프로젝트는 교육용 가상 부대·가상 품목·가상 저장고 자료를 코드로 만들어, 탄약·유류 소모율 데이터 정의와 전처리, CSR 기반 Baseline과 탐색적 분석, 적 저항·화력 계획을 반영한 소모량 예측, 보급 수준(DOS) 연계 재보급 산출, 작전 지도 대시보드와 설명가능성, 저장 배치 최적화와 군수참모 결재선(HITL), 7일 고강도 스트레스 시험, AAR 기반 소모 계수 재보정을 단계별로 구현한다. 최종 목표는 예측 소모량을 근거로 재보급 소요와 저장 위치를 함께 제안해 저장 효율과 작업 동선을 개선하고, 불출·대체 탄종·재배치는 군수참모가 근거를 확인해 결정하도록 지원하는 통합 시뮬레이터를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(군수참모처, 탄약창)에 즉시 도입 가능한 "완성형 국방 AI 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 행정 및 기술 명세서** | 도메인 데이터 명세서(`docs/amm02_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/amm02_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **군수 데이터 정제 모듈** | 10종 정제 룰 및 긴급 불출 보존 ETL 파이프라인(`src/etl/amm02_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **분석 및 통계 검증** | 90일 소모량 Baseline 비교, 가설 검정 및 ABC 파레토 분석(`notebooks/amm02_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 시계열 파생 피처 엔지니어링 모듈(`src/features/amm02_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 재보급 엔진** | GBR+RF 앙상블 및 DOS 재보급 산출 엔진(`src/models/amm02_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | XAI 기여도 분석 및 자연어 보고서 생성기(`src/explainability/amm02_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **저장 최적화 & HITL** | 정수계획 저장 최적화 및 참모 HITL 결재선 엔진(`src/optimization/amm02_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **작전 지도 대시보드** | 전술 격자 지도 연동 인터랙티브 대시보드 웹앱(`src/dashboard/amm02_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 편차 분석, 축소 추정 CSR 재보정 및 3대 정량 효과 리포트(`reports/amm02_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 훈련 데이터 기반 챔피언/챌린저 모델 자동 성능 비교 및 롤오버 파이프라인(`pipeline/amm02_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 교전 일수 기반 1차 탄약/유류 기본 소모량 계산 Baseline 알고리즘 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

교전 일수 기반 1차 탄약/유류 기본 소모량 계산 Baseline 알고리즘을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **교전 일수 x 작전 단계별 CSR x 병력 규모로 1차 탄약·유류 기본 소모량을 계산하는 Baseline 알고리즘을 구현하고 탐색적 분석으로 한계를 찾는다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **부대별 90일 계획량과 실제 소모량을 비교하고 품목별 일일 MAPE 산출하고 모델에 적용하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `notebooks/amm02_eda_analysis.ipynb 및 EDA 요약 리포트` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 의사결정에 직관적인 인사이트 제공 수준

#### 미션 PASS 조건

- 정량 기준: EDA 시각화 차트 8종 이상 생성, 유의미한 상관관계 피처 5개 이상 도출


- 진행 기준: EDA 노트북 실행 완료 및 핵심 발견점(Key Findings) 문서화


- 검증 방법: 통계적 가설 검정(p-value, 상관계수)을 통한 유의성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""교전 일수 x 작전 단계별 기준 소모율(CSR) x 병력 규모로 1차 탄약·유류 기본 소모량을 계산하는 Baseline 알고리즘을 구현하고, 탐색적 분석(EDA 차트 8종, 상관·유의성 검정, ABC 파레토 분석)으로 Baseline의 한계와 핵심 변수를 찾는다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 부대·가상 품목 일일 소모 기록 생성기 (실제 탄약 제원·소모율과 무관)
# 가상 부대 8개 x 90일. 소모량 = 가상 CSR 계수 x 병력 규모 x (교전 강도·적 저항·화력 계획 효과) x 잡음
# ---------------------------------------------------------------------------
UNITS = [f"U{i:02d}" for i in range(1, 9)]
ITEMS = ["ammo_A", "ammo_B", "ammo_C", "ammo_D", "fuel_jet", "fuel_diesel"]
AMMO, FUEL = ITEMS[:4], ITEMS[4:]
PHASES = ["prep", "attack", "defense", "stability"]
CSR = {  # 가상 계수: 병력 100명·1일 기준 소모 단위(탄약=상자, 유류=드럼)
    "prep":      [1.0, 0.6, 0.3, 0.2, 1.5, 6.0],
    "attack":    [6.0, 3.5, 2.0, 1.2, 5.0, 12.0],
    "defense":   [4.5, 3.0, 1.6, 0.8, 3.0, 7.0],
    "stability": [0.8, 0.5, 0.3, 0.2, 1.0, 5.0],
}
PHASE_RESIST = {"prep": 1.0, "attack": 3.2, "defense": 3.6, "stability": 0.8}


def make_ops(n_days=90, seed=7, surge=0):
    """가상 부대별 일일 작전 기록과 품목별 실제 소모량(정답)을 만든다. surge>0이면 마지막 surge일을 고강도 공격 단계로 바꾼다(스트레스 시험용)."""
    rng = np.random.default_rng(seed)
    rows = []
    for uid in UNITS:
        size = int(rng.integers(3, 9)) * 100
        lens = rng.integers(8, 20, 4)
        sched = ["prep"] * lens[0] + ["attack"] * lens[1] + ["defense"] * lens[2] + ["attack"] * lens[3]
        sched = (sched + ["stability"] * n_days)[:n_days]
        if surge:
            sched[-surge:] = ["attack"] * surge
        r = 1.0
        for d, ph in enumerate(sched):
            r = float(np.clip(0.7 * r + 0.3 * (4.8 if surge and d >= n_days - surge else PHASE_RESIST[ph]) + rng.normal(0, 0.35), 0, 5))
            inten = float(np.clip(r / 5 * 0.9 + rng.normal(0, 0.06), 0, 1))
            fire = int(rng.poisson(1 + 8 * inten))
            move = float(rng.gamma(2, {"prep": 4, "attack": 12, "defense": 2, "stability": 5}[ph]))
            weather = str(rng.choice(["clear", "rain", "snow"], p=[0.7, 0.2, 0.1]))
            emerg = bool(rng.random() < 0.02)
            rec = {"unit": uid, "day": d, "phase": ph, "unit_size": size, "intensity": round(inten, 3),
                   "resistance": round(r, 2), "fire_missions": fire, "move_km": round(move, 1),
                   "weather": weather, "emergency": emerg}
            base = size / 100
            for i, it in enumerate(ITEMS):
                c = CSR[ph][i] * base
                if it in AMMO:
                    c *= (0.5 + 0.6 * inten) * (1 + 0.04 * fire * (1.0 if it in ("ammo_C", "ammo_D") else 0.4))
                elif it == "fuel_jet":
                    c *= (0.6 + 0.5 * inten) + 0.03 * fire
                else:
                    c = c * (0.8 + 0.2 * (weather != "clear")) + 0.05 * move * base
                c *= rng.lognormal(0, 0.12) * (2.5 if emerg else 1.0)
                rec[it] = round(float(c), 1)
            rows.append(rec)
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------


def baseline(df):
    """Baseline: 해당 일의 작전 단계 CSR x (병력/100). 교전 강도·적 저항·화력 계획은 반영하지 않는다."""
    # [TODO : 학생 실습 3] 교전 일수 기반 1차 탄약/유류 기본 소모량 계산 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
df = make_ops(90, seed=7)
base = baseline(df)

# 1) 교전 일수 기반 부대별 작전 기간 총소모량(계획) vs 실제
df["engaged"] = df["phase"].isin(["attack", "defense"])
plan = pd.concat([df[["unit"]], base], axis=1).groupby("unit").sum()
real = df.groupby("unit")[ITEMS].sum()
days = df.groupby("unit")["engaged"].sum()
tot = pd.DataFrame({"교전일수": days, "탄약 계획": plan[AMMO].sum(axis=1).round(), "탄약 실제": real[AMMO].sum(axis=1).round(),
                    "유류 계획": plan[FUEL].sum(axis=1).round(), "유류 실제": real[FUEL].sum(axis=1).round()})
tot["탄약 오차율"] = ((tot["탄약 계획"] - tot["탄약 실제"]) / tot["탄약 실제"]).map("{:+.1%}".format)
print("[부대별 90일 총소모량: Baseline 계획 vs 실제(가상 단위)]")
print(tot.to_string())

# 2) 일일 예측 오차(MAPE)
mape = (np.abs(base - df[ITEMS]) / df[ITEMS]).mean()
print("\nBaseline 일일 MAPE:", ", ".join(f"{k} {v:.1%}" for k, v in mape.items()))

# 3) 상관·유의성 검정: 병력 100명당 탄약 소모율과 각 변수
df["ammo_rate"] = df[AMMO].sum(axis=1) / (df["unit_size"] / 100)
df["bad_weather"] = (df["weather"] != "clear").astype(int)
cand = ["intensity", "resistance", "fire_missions", "move_km", "engaged", "day", "bad_weather", "unit_size"]
rows = []
for c in cand:
    r, p = stats.pearsonr(df[c].astype(float), df["ammo_rate"])
    rows.append({"변수": c, "상관계수": round(r, 3), "p-value": f"{p:.1e}", "유의(p<0.05,|r|>=0.2)": p < 0.05 and abs(r) >= 0.2})
corr = pd.DataFrame(rows).sort_values("상관계수", key=abs, ascending=False)
n_sig = int(corr.iloc[:, 3].sum())
print("\n[탄약 환산 소모율과의 상관·유의성]")
print(corr.to_string(index=False))
print(f"유의미한 상관 변수 {n_sig}개 (목표 5개 이상: {'달성' if n_sig >= 5 else '미달'})")

# 4) ABC 파레토 분석: 부대 x 탄종 조합별 총소모량(상자) 상위 20%
pareto = real[AMMO].stack().sort_values(ascending=False)
top = pareto.iloc[:int(np.ceil(len(pareto) * 0.2))]
print(f"\nABC 분석: 상위 20% 조합({len(top)}개)이 전체 소모량의 {top.sum() / pareto.sum():.1%}를 차지 -> A등급 집중 관리")
print("A등급 예시:", ", ".join(f"{u}-{i}" for u, i in top.index[:5]))

# 5) EDA 차트 8종
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
ax = ax.ravel()
df.groupby("phase")[AMMO].mean().plot.bar(ax=ax[0], title="1. Mean ammo use by phase")
ax[1].scatter(df["intensity"], df["ammo_rate"], s=6, alpha=0.5); ax[1].set_title("2. Intensity vs ammo rate")
ax[2].scatter(df["fire_missions"], df["ammo_rate"], s=6, alpha=0.5, c="tab:red"); ax[2].set_title("3. Fire missions vs ammo rate")
ax[3].scatter(df["move_km"], df["fuel_diesel"] / (df["unit_size"] / 100), s=6, alpha=0.5, c="tab:green"); ax[3].set_title("4. Move km vs diesel rate")
u = df[df["unit"] == "U01"]
ax[4].plot(u["day"], u[AMMO].sum(axis=1), label="actual"); ax[4].plot(u["day"], base.loc[u.index, AMMO].sum(axis=1), label="baseline")
ax[4].legend(); ax[4].set_title("5. U01 daily ammo: actual vs baseline")
im = ax[5].imshow(df[cand + ["ammo_rate"]].astype(float).corr(), cmap="coolwarm", vmin=-1, vmax=1)
ax[5].set_xticks(range(9)); ax[5].set_xticklabels(cand + ["ammo_rate"], rotation=90, fontsize=7); ax[5].set_yticks([])
ax[5].set_title("6. Correlation heatmap")
ax[6].bar(range(len(pareto)), pareto.cumsum() / pareto.sum()); ax[6].axhline(0.8, ls="--", c="k"); ax[6].set_title("7. Pareto (unit x ammo item)")
ax[7].bar(mape.index, mape.values, color="gray"); ax[7].set_title("8. Baseline MAPE by item"); ax[7].tick_params(axis="x", rotation=45)
plt.suptitle("AMM02 EDA - synthetic consumption data"); plt.tight_layout(); plt.show()
print("EDA 차트 8종 생성 (목표 8종 이상: 달성)")
print("핵심 발견: Baseline은 교전 강도·화력 계획을 반영하지 않아 고강도 일에는 과소, 저강도 일에는 과대 산출한다 -> 미션 4에서 변수 반영 모델로 고도화한다.")


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[부대별 90일 총소모량: Baseline 계획 vs 실제(가상 단위)]
      교전일수   탄약 계획   탄약 실제   유류 계획   유류 실제  탄약 오차율
unit                                              
U01     48  5114.0  5120.0  7716.0  7959.0   -0.1%
U02     40  3952.0  3885.0  6328.0  6678.0   +1.7%
U03     46  3725.0  3493.0  5715.0  5616.0   +6.6%
U04     41  2893.0  2586.0  4618.0  4504.0  +11.9%
U05     36  2139.0  1971.0  3574.0  3355.0   +8.5%
U06     34  4028.0  3598.0  6820.0  6404.0  +12.0%
U07     26  2945.0  2578.0  5408.0  5282.0  +14.2%
U08     49  4541.0  4124.0  6776.0  6643.0  +10.1%

Baseline 일일 MAPE: ammo_A 43.8%, ammo_B 42.9%, ammo_C 38.1%, ammo_D 37.9%, fuel_jet 25.9%, fuel_diesel 16.8%

[탄약 환산 소모율과의 상관·유의성]
           변수   상관계수  p-value  유의(p<0.05,|r|>=0.2)
      engaged  0.923 1.4e-300                 True
   resistance  0.854 9.2e-206                 True
    intensity  0.838 1.1e-190                 True
fire_missions  0.639  7.9e-84                 True
          day -0.378  6.9e-26                 True
      move_km  0.294  7.5e-16                 True
  bad_weather  0.072  5.2e-02                False
    unit_size  0.017  6.4e-01                False
유의미한 상관 변수 6개 (목표 5개 이상: 달성)

ABC 분석: 상위 20% 조합(7개)이 전체 소모량의 43.0%를 차지 -> A등급 집중 관리
A등급 예시: U01-ammo_A, U08-ammo_A, U02-ammo_A, U06-ammo_A, U03-ammo_A
EDA 차트 8종 생성 (목표 8종 이상: 달성)
핵심 발견: Baseline은 교전 강도·화력 계획을 반영하지 않아 고강도 일에는 과소, 저강도 일에는 과대 산출한다 -> 미션 4에서 변수 반영 모델로 고도화한다.
```
</details>
